<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/TIC-TAC-TOE_lektioner/Lab_2_Lektion_5_Sokalgoritmerna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Tre i rad – Lektion 5: Sökalgoritmerna (DFS & BFS)

**Målgrupp:** Gymnasiet, inga förkunskaper i programmering krävs  
**Tid:** ca 60 minuter  
**Mål:** Förstå sökalgoritmerna DFS och BFS, jämföra dem med regelbaserad AI, och se varför sökning är ett kraftfullt verktyg

---

**Upphovspersoner:** LiU / Tekniksprånget  
**Licens:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.sv)

---

### 📚 Förutsättning
Den här lektionen bygger vidare på **Lektion 4**.  
Du ska känna till `random_agent()` och `rule_based_agent()` innan du fortsätter.

### 🗺️ Vad gör vi idag?
1. Förstår **problemet** med regelbaserade agenter 🤔
2. Lär oss vad ett **spelträd** är 🌳
3. Implementerar **DFS** – Djupet-först sökning 🔍
4. Implementerar **BFS** – Bredden-först sökning 🌊
5. **Jämför** DFS och BFS mot varandra och mot regelbaserad agent 📊
6. Löser **övningar** och tar ett **quiz** 📝


In [ ]:
# ============================================================
# INSTÄLLNINGAR – Kör denna cell FÖRST!
# Innehåller all spellogik från tidigare lektioner.
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import copy
import time
from collections import deque

BOARD_SIZE = 3   # Spelplanen är alltid 3x3

# ----------------------------------------------------------
# check_winner – kollar om någon spelare vann
# Returnerar: 1 om spelare 1 vann, 2 om spelare 2 vann,
#             3 om det är oavgjort (fullt bräde), 0 om spelet pågår
# ----------------------------------------------------------
def check_winner(board):
    """Returnerar 1, 2, 3 (oavgjort) eller 0 (spelet pågår)."""
    # Kolla rader
    for rad in range(BOARD_SIZE):
        if board[rad][0] != 0 and board[rad][0] == board[rad][1] == board[rad][2]:
            return board[rad][0]
    # Kolla kolumner
    for kol in range(BOARD_SIZE):
        if board[0][kol] != 0 and board[0][kol] == board[1][kol] == board[2][kol]:
            return board[0][kol]
    # Kolla diagonaler
    if board[0][0] != 0 and board[0][0] == board[1][1] == board[2][2]:
        return board[0][0]
    if board[0][2] != 0 and board[0][2] == board[1][1] == board[2][0]:
        return board[0][2]
    # Kolla om brädet är fullt (oavgjort)
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 3  # Oavgjort
    return 0  # Spelet pågår

# ----------------------------------------------------------
# make_move – gör ett drag om rutan är tom
# ----------------------------------------------------------
def make_move(board, row, col, player):
    """Returnerar True om draget lyckades, False annars."""
    if board[row][col] == 0:
        board[row][col] = player
        return True
    return False

# ----------------------------------------------------------
# visa_braede – skriver ut spelplanen som text
# ----------------------------------------------------------
def visa_braede(board):
    """Skriver ut spelplanen med symboler i terminalen."""
    symboler = {0: ".", 1: "X", 2: "O"}
    print("  0 1 2")
    for rad in range(BOARD_SIZE):
        rad_text = f"{rad} "
        for kol in range(BOARD_SIZE):
            rad_text += symboler[board[rad][kol]] + " "
        print(rad_text)
    print()

# ----------------------------------------------------------
# find_empty_cells – returnerar alla tomma celler
# ----------------------------------------------------------
def find_empty_cells(board):
    """Returnerar lista med tomma (rad, kol) positioner."""
    tomma = []
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:
                tomma.append((rad, kol))
    return tomma

# ----------------------------------------------------------
# random_agent – väljer ett slumpmässigt drag
# ----------------------------------------------------------
def random_agent(board, player=None):
    """Väljer ett slumpmässigt drag bland de tomma cellerna."""
    tomma_celler = find_empty_cells(board)
    if tomma_celler:
        return random.choice(tomma_celler)
    return None

# ----------------------------------------------------------
# rule_based_agent – regelbaserad agent från Lektion 4
# ----------------------------------------------------------
def rule_based_agent(board, player):
    """Regelbaserad agent: Vinn, Blockera, Ta mitten, Ta hörn, Ta vad som helst."""
    motstandare = 2 if player == 1 else 1

    # Regel 1: Vinn om möjligt
    for (rad, kol) in find_empty_cells(board):
        board[rad][kol] = player
        if check_winner(board) == player:
            board[rad][kol] = 0
            return (rad, kol)
        board[rad][kol] = 0

    # Regel 2: Blockera motspelaren
    for (rad, kol) in find_empty_cells(board):
        board[rad][kol] = motstandare
        if check_winner(board) == motstandare:
            board[rad][kol] = 0
            return (rad, kol)
        board[rad][kol] = 0

    # Regel 3: Ta mitten
    if board[1][1] == 0:
        return (1, 1)

    # Regel 4: Ta ett hörn
    for (rad, kol) in [(0,0), (0,2), (2,0), (2,2)]:
        if board[rad][kol] == 0:
            return (rad, kol)

    # Annars: Ta vad som helst
    return random_agent(board)

# ----------------------------------------------------------
# play_game – kör ett spel mellan två agenter
# ----------------------------------------------------------
def play_game(agent1, agent2, verbose=False):
    """
    Spelar ett komplett spel mellan agent1 och agent2.
    Returnerar vinnaren: 1, 2 eller 3 (oavgjort).
    """
    bräde = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    agenter = [agent1, agent2]
    spelare = [1, 2]

    for drag_nummer in range(BOARD_SIZE * BOARD_SIZE):
        aktuell_agent = agenter[drag_nummer % 2]
        aktuell_spelare = spelare[drag_nummer % 2]

        drag = aktuell_agent(bräde, aktuell_spelare)
        if drag is None:
            break
        make_move(bräde, drag[0], drag[1], aktuell_spelare)

        if verbose:
            visa_braede(bräde)

        resultat = check_winner(bräde)
        if resultat != 0:
            return resultat

    return 3  # Oavgjort

# ----------------------------------------------------------
# run_tournament – kör många spel och räknar statistik
# ----------------------------------------------------------
def run_tournament(agent1, agent2, antal_spel=100):
    """
    Kör ett turneringsläge med många spel.
    Returnerar (vinster_agent1, vinster_agent2, oavgjorda).
    """
    vinster_1 = 0
    vinster_2 = 0
    oavgjorda = 0

    for _ in range(antal_spel):
        resultat = play_game(agent1, agent2)
        if resultat == 1:
            vinster_1 += 1
        elif resultat == 2:
            vinster_2 += 1
        else:
            oavgjorda += 1

    return vinster_1, vinster_2, oavgjorda

print("✅ Alla funktioner från tidigare lektioner är laddade!")
print("🎮 Redo att lära oss om sökalgoritmerna!")


## 🤔 Del 1: Problemet med regelbaserade AI

### Vad kan regelbaserad AI?

I Lektion 4 byggde vi en regelbaserad agent. Den var ganska smart:
- ✅ **Vinner** om det finns ett direkt vinnardrag
- ✅ **Blockerar** om motståndaren är nära att vinna
- ✅ **Tar mitten** som en smart position

### Men vad kan den INTE?

Tänk dig det här scenariot:

```
  0 1 2
0 X . O
1 . X .
2 . . O
```

Motståndaren (O) kan **inte** vinna direkt – men de planerar ett **gaffeldrag**!  
En regelbaserad agent ser inte det.

### 🧠 Konceptet: Planera framåt

Istället för regler, kan vi **söka** genom alla möjliga framtida drag?

> 💡 **Sökalgoritmerna** bygger ett **träd av möjliga framtida drag**  
> och väljer det drag som leder till bäst resultat!

Det är precis som att spela schack i huvudet – du tänker igenom:  
*"Om jag gör X, kan motståndaren göra Y, och då kan jag göra Z..."*


## 🌳 Del 2: Vad är ett spelträd?

### Trädet av möjligheter

Varje nod i trädet är ett **tillstånd** (hur spelplanen ser ut).  
Varje kant är ett **drag** (ett steg från ett tillstånd till nästa).

```
           Tom spelplan
                |
    ____________|____________
    |           |            |
  X på (0,0)  X på (0,1)  X på (0,2) ...  ← Nivå 1 (9 möjligheter)
    |
  ____|____
  |       |
O på (0,1) O på (0,2) ...               ← Nivå 2 (8 möjligheter)
  |
...                                      ← Nivå 3 (7 möjligheter)
```

### 📊 Hur stort är trädet?

För Tre i rad:
- **Nivå 1**: 9 möjliga drag
- **Nivå 2**: 8 möjliga drag
- **Nivå 3**: 7 möjliga drag
- ...osv

**Totalt**: upp till 9! = 9 × 8 × 7 × ... × 1 = **362 880 möjliga spel**

Det låter mycket, men för en dator är det litet! 💻

### Två sätt att söka i trädet

| Algoritm | Strategi | Liknelse |
|----------|----------|----------|
| **DFS** (Djupet-först) | Gå djupt innan du är bred | Utforska en labyrint rakt fram |
| **BFS** (Bredden-först) | Utforska all bredd före djup | Ringar på vatten |


## 🔍 Del 3: DFS – Djupet-först sökning

### Vad är DFS?

**DFS** (Depth-First Search) = Djupet-först sökning

> 🗺️ **Analogin:** DFS är som att utforska en labyrint.  
> Du går alltid rakt fram, och backar bara när du kör fast.  
> Sedan provar du nästa väg.

### Hur DFS fungerar i ett spelträd:

```
Spelplan:
  0 1 2
0 X O .
1 . X .
2 . . O

DFS väljer drag (0,2) och söker neråt:
  → Spelar (0,2), kollar om X vann → Nej
    → Nästa spelare spelar (1,0), söker neråt:
      → Spelar (2,0), kollar om O vann → Nej
        → Spelar (1,1)... osv till slutet av spelet
          → OAVGJORT! Backar...
        → Spelar (1,2)... osv
      ...
    → Backar och provar (1,2) osv...
  → Backar och provar (1,0) osv...
```

### 💡 Nyckelkonceptet: Rekursion

DFS använder sig av **rekursion** – en funktion som anropar sig själv!  
Varje anrop av funktionen = ett djupare steg i trädet.


In [ ]:
# ============================================================
# DFS-AGENTEN – Djupet-först sökning
# ============================================================

def dfs_agent(board, player):
    """
    DFS-agent: Letar igenom spelträdet på djupet.
    Returnerar det bästa draget för spelaren.
    """

    def dfs(bräde, spelare, djup=0):
        """
        Rekursiv DFS-funktion.

        bräde  = aktuell spelplan
        spelare = vems tur det är (1 eller 2)
        djup   = hur djupt vi är i trädet (för felsökning)
        """

        # --- BASFALLET: Kolla om spelet är klart ---
        vinnare = check_winner(bräde)

        if vinnare == player:
            return 1, None    # ✅ Vi vinner!
        elif vinnare != 0 and vinnare != player:
            return -1, None   # ❌ Vi förlorar!
        elif vinnare == 3:
            return 0, None    # 🤝 Oavgjort

        # --- REKURSIVT FALL: Prova alla möjliga drag ---
        bästa_poäng = -2    # Startpoäng (värre än möjligt)
        bästa_drag = None

        for (rad, kol) in find_empty_cells(bräde):
            # Gör draget
            bräde[rad][kol] = spelare

            # Nästa spelare
            nästa_spelare = 2 if spelare == 1 else 1

            # Rekursivt anrop: kolla vad som händer framöver
            poäng, _ = dfs(bräde, nästa_spelare, djup + 1)

            # Invertera poäng – bra för motståndaren = dåligt för oss
            poäng = -poäng

            # Ångra draget (backtracka)
            bräde[rad][kol] = 0

            # Spara bästa drag
            if poäng > bästa_poäng:
                bästa_poäng = poäng
                bästa_drag = (rad, kol)

        return bästa_poäng, bästa_drag

    # Starta sökningen och returnera det bästa draget
    _, drag = dfs(board, player)
    return drag

print("✅ dfs_agent() är redo!")
print()
print("🧪 Testar DFS på en enkel spelplan...")
test_bräde = [
    [1, 1, 0],   # X X .  ← X kan vinna på (0,2)!
    [2, 0, 0],   # O . .
    [0, 0, 2]    # . . O
]
visa_braede(test_bräde)
drag = dfs_agent(test_bräde, 1)
print(f"DFS väljer drag: {drag}")
print("(Rätt svar: (0, 2) – X vinner direkt!)")


### 🧪 Experiment: Kör DFS mot slumpmässig agent

Nu testar vi hur bra DFS är! Låt oss köra 50 spel och se statistiken.


In [ ]:
# ============================================================
# EXPERIMENT: DFS vs Slumpmässig agent
# ============================================================

print("⏱️  Mäter tid för DFS...")
start_tid = time.time()

# Kör 50 spel – DFS som spelare 1 mot slumpmässig som spelare 2
vinster_dfs, vinster_slump, oavgjorda = run_tournament(dfs_agent, random_agent, antal_spel=50)

slut_tid = time.time()
total_tid = slut_tid - start_tid

print(f"\n📊 Resultat efter 50 spel (DFS vs Slumpmässig):")
print(f"   DFS vann:          {vinster_dfs} gånger ({vinster_dfs*2}%)")
print(f"   Slumpmässig vann:  {vinster_slump} gånger ({vinster_slump*2}%)")
print(f"   Oavgjorda:         {oavgjorda} gånger ({oavgjorda*2}%)")
print(f"\n⏱️  Total tid: {total_tid:.2f} sekunder")
print(f"   Tid per spel: {total_tid/50*1000:.1f} millisekunder")
print()
if vinster_dfs >= 40:
    print("🌟 Imponerande! DFS är klart bättre än slump!")
elif vinster_dfs >= 25:
    print("👍 DFS är bättre än slump, men inte perfekt.")
else:
    print("🤔 Oväntat resultat – kolla koden!")


## 🌊 Del 4: BFS – Bredden-först sökning

### Vad är BFS?

**BFS** (Breadth-First Search) = Bredden-först sökning

> 💧 **Analogin:** BFS är som att kasta en sten i vatten.  
> Ringarna breder ut sig **jämnt** i alla riktningar, nivå för nivå.  
> Inte som DFS som dyker rakt ner!

### Hur BFS fungerar:

```
Träd-nivåer BFS utforskar:

Nivå 0: [Ursprungstillstånd]
                |
Nivå 1: [Drag A]  [Drag B]  [Drag C] ...    ← Alla utforskas FÖRST
                |
Nivå 2: [A1][A2]  [B1][B2]  [C1][C2] ...   ← Sedan dessa
                |
Nivå 3: ...                                  ← Sedan dessa, osv
```

### 📦 Datastrukturen: Kön (Queue)

BFS använder en **kö** (FIFO – First In, First Out):
- Du lägger till nya tillstånd **i slutet** av kön
- Du tar ut tillstånd **från fronten** av kön
- Python har `collections.deque` för effektiv kö

```
Kö: [Tillstånd A, Tillstånd B, Tillstånd C]
     ↑ Ta härifrån             ↑ Lägg till här
```

> 💡 **Varför kö för BFS?** Kön garanterar att vi utforskar alla tillstånd på  
> samma djupnivå innan vi går djupare. Det är definitionen av BFS!


In [ ]:
# ============================================================
# BFS-AGENTEN – Bredden-först sökning
# ============================================================

def bfs_agent(board, player):
    """
    BFS-agent: Utforskar alla möjliga drag nivå för nivå.
    Returnerar det bästa draget för spelaren.
    """

    # --- Starta med en kö ---
    # Varje element i kön: (bräde, första_drag, aktuell_spelare)
    # 'första_drag' = vilket drag VI gör i första steget
    kö = deque()

    # Lägg till alla möjliga FÖRSTA drag i kön
    for (rad, kol) in find_empty_cells(board):
        nytt_bräde = copy.deepcopy(board)   # Kopiera brädet
        nytt_bräde[rad][kol] = player        # Gör draget
        kö.append((nytt_bräde, (rad, kol), player))

    bästa_drag = None  # Standarddrag om inget vinnande hittades

    # --- Utforska trädet på bredden ---
    while kö:
        # Ta nästa tillstånd från fronten av kön
        aktuellt_bräde, första_drag, aktuell_spelare = kö.popleft()

        # Kolla om spelet är klart
        vinnare = check_winner(aktuellt_bräde)

        if vinnare == player:
            return första_drag   # 🏆 Hittade ett vinnande drag!

        if vinnare != 0:
            continue   # Spelet är klart (förlust eller oavgjort), hoppa över

        # Lägg till nästa nivåns tillstånd i kön
        nästa_spelare = 2 if aktuell_spelare == 1 else 1

        for (rad, kol) in find_empty_cells(aktuellt_bräde):
            nytt_bräde = copy.deepcopy(aktuellt_bräde)
            nytt_bräde[rad][kol] = nästa_spelare
            kö.append((nytt_bräde, första_drag, nästa_spelare))

        # Kom ihåg ett drag om inget vinnande hittas
        if bästa_drag is None:
            bästa_drag = första_drag

    # Returnera det bästa draget (eller ett slumpmässigt om kön är tom)
    tomma = find_empty_cells(board)
    return bästa_drag if bästa_drag else (tomma[0] if tomma else None)

print("✅ bfs_agent() är redo!")
print()
print("🧪 Testar BFS på samma spelplan som DFS...")
test_bräde = [
    [1, 1, 0],   # X X .  ← X kan vinna på (0,2)!
    [2, 0, 0],   # O . .
    [0, 0, 2]    # . . O
]
visa_braede(test_bräde)
drag = bfs_agent(test_bräde, 1)
print(f"BFS väljer drag: {drag}")
print("(Rätt svar: (0, 2) – X vinner direkt!)")


## 📊 Del 5: Jämförelse – DFS vs BFS

Dags att jämföra algoritmerna! Vi mäter:
1. **Vinstprocent** mot slumpmässig agent
2. **Hastighet** (tid per spel)
3. **Strategi** (hur de tänker annorlunda)


In [ ]:
# ============================================================
# STOR JÄMFÖRELSE: DFS vs BFS vs Regelbaserad vs Slumpmässig
# ============================================================

print("🏁 Startar jämförelseturnering (50 spel vardera)...")
print("=" * 55)

# --- DFS vs Slumpmässig ---
start = time.time()
v1, v2, o = run_tournament(dfs_agent, random_agent, 50)
tid_dfs = time.time() - start
print(f"\n🔍 DFS vs Slumpmässig:")
print(f"   DFS vann: {v1}/50  |  Slump vann: {v2}/50  |  Oavgjort: {o}/50")
print(f"   ⏱️  Tid: {tid_dfs:.2f}s ({tid_dfs/50*1000:.0f}ms/spel)")

# --- BFS vs Slumpmässig ---
start = time.time()
v1, v2, o = run_tournament(bfs_agent, random_agent, 50)
tid_bfs = time.time() - start
print(f"\n🌊 BFS vs Slumpmässig:")
print(f"   BFS vann: {v1}/50  |  Slump vann: {v2}/50  |  Oavgjort: {o}/50")
print(f"   ⏱️  Tid: {tid_bfs:.2f}s ({tid_bfs/50*1000:.0f}ms/spel)")

# --- Regelbaserad vs Slumpmässig ---
start = time.time()
v1, v2, o = run_tournament(rule_based_agent, random_agent, 50)
tid_regel = time.time() - start
print(f"\n📋 Regelbaserad vs Slumpmässig:")
print(f"   Regelbaserad vann: {v1}/50  |  Slump vann: {v2}/50  |  Oavgjort: {o}/50")
print(f"   ⏱️  Tid: {tid_regel:.2f}s ({tid_regel/50*1000:.0f}ms/spel)")

# --- DFS vs BFS ---
start = time.time()
v1, v2, o = run_tournament(dfs_agent, bfs_agent, 50)
tid_dvb = time.time() - start
print(f"\n⚔️  DFS vs BFS:")
print(f"   DFS vann: {v1}/50  |  BFS vann: {v2}/50  |  Oavgjort: {o}/50")
print(f"   ⏱️  Tid: {tid_dvb:.2f}s")

print("\n" + "=" * 55)
print("📈 Sammanfattning:")
print(f"   DFS är {tid_dfs/tid_regel:.1f}x långsammare än regelbaserad")
print(f"   BFS är {tid_bfs/tid_regel:.1f}x långsammare än regelbaserad")
print()
print("💡 Sökning är kraftfull men tar mer tid!")
print("💡 I nästa lektion lär vi oss Minimax – ännu smartare!")


## 🎮 Del 6: Spela mot DFS-agenten!

Nu är det dags att utmana DFS-agenten!  
Klicka på knapparna nedan för att göra dina drag.  
Du spelar som **O** (spelare 2), DFS-agenten spelar som **X** (spelare 1).


In [ ]:
# ============================================================
# INTERAKTIVT SPEL MOT DFS-AGENTEN
# ============================================================

def spela_interaktivt(ai_agent, ai_namn="DFS"):
    """Skapar ett interaktivt spel mot en AI-agent."""

    bräde = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    spel_slut = [False]  # Lista för att kunna ändra inuti nested functions

    symboler = {0: "⬜", 1: "❌", 2: "⭕"}
    knappar = [[None]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    status_label = widgets.Label(value=f"🎮 Du spelar mot {ai_namn}! Du är ⭕ (O)")

    def uppdatera_display():
        """Uppdaterar knapparna baserat på brädet."""
        for r in range(BOARD_SIZE):
            for k in range(BOARD_SIZE):
                knappar[r][k].description = symboler[bräde[r][k]]
                knappar[r][k].disabled = (bräde[r][k] != 0) or spel_slut[0]

    def ai_drag():
        """Låter AI-agenten göra sitt drag."""
        drag = ai_agent(bräde, 1)  # AI spelar som spelare 1 (X)
        if drag:
            make_move(bräde, drag[0], drag[1], 1)
            uppdatera_display()
            vinnare = check_winner(bräde)
            if vinnare == 1:
                status_label.value = f"❌ {ai_namn} vann! Bättre lycka nästa gång!"
                spel_slut[0] = True
                for r in range(BOARD_SIZE):
                    for k in range(BOARD_SIZE):
                        knappar[r][k].disabled = True
            elif vinnare == 3:
                status_label.value = "�� Oavgjort! Bra spelat!"
                spel_slut[0] = True
                for r in range(BOARD_SIZE):
                    for k in range(BOARD_SIZE):
                        knappar[r][k].disabled = True

    def knapp_klickad(b, rad, kol):
        """Hanterar ett klick på en knapp."""
        if spel_slut[0] or bräde[rad][kol] != 0:
            return
        # Spelarens drag (spelare 2 = O)
        make_move(bräde, rad, kol, 2)
        uppdatera_display()
        vinnare = check_winner(bräde)
        if vinnare == 2:
            status_label.value = "⭕ Du vann! Grattis! 🎉"
            spel_slut[0] = True
            return
        elif vinnare == 3:
            status_label.value = "🤝 Oavgjort! Bra spelat!"
            spel_slut[0] = True
            return
        status_label.value = f"🤔 {ai_namn} tänker..."
        ai_drag()
        if not spel_slut[0]:
            status_label.value = "🎮 Din tur! Du är ⭕"

    # Bygg GUI
    rader_widgets = []
    for r in range(BOARD_SIZE):
        rad_knappar = []
        for k in range(BOARD_SIZE):
            knapp = widgets.Button(
                description=symboler[0],
                layout=widgets.Layout(width="60px", height="60px"),
                style={"button_color": "#f0f0f0"}
            )
            knapp.on_click(lambda b, rad=r, kol=k: knapp_klickad(b, rad, kol))
            knappar[r][k] = knapp
            rad_knappar.append(knapp)
        rader_widgets.append(widgets.HBox(rad_knappar))

    nytt_spel_knapp = widgets.Button(
        description="🔄 Nytt spel",
        button_style="info",
        layout=widgets.Layout(width="120px")
    )

    def nytt_spel(b):
        """Återställer spelet."""
        for r in range(BOARD_SIZE):
            for k in range(BOARD_SIZE):
                bräde[r][k] = 0
        spel_slut[0] = False
        uppdatera_display()
        status_label.value = f"🎮 Du spelar mot {ai_namn}! Du är ⭕ (O)"

    nytt_spel_knapp.on_click(nytt_spel)

    display(widgets.VBox([
        status_label,
        *rader_widgets,
        nytt_spel_knapp,
        widgets.Label(value="💡 Tips: DFS tänker igenom hela spelträdet!")
    ]))

# Starta spelet mot DFS
spela_interaktivt(dfs_agent, "DFS-agenten 🔍")


## 🌊 Spela mot BFS-agenten!

Testa nu BFS-agenten – märker du någon skillnad mot DFS?


In [ ]:
# Spela mot BFS-agenten
spela_interaktivt(bfs_agent, "BFS-agenten 🌊")


## ✏️ Del 7: Övningar

Arbeta igenom dessa övningar för att fördjupa din förståelse!

### 🟢 Grundläggande övningar (Enkla)


### Övning 1: Spåra DFS

Titta på denna spelplan:
```
  0 1 2
0 X . .
1 . O .
2 . . X
```

**Fråga:** Om DFS är X (spelare 1) och det är X:s tur, vilka drag kan DFS prova?  
Lista dem i ordning (from vänster till höger, uppifrån och ner):

*(Ledtråd: titta på `find_empty_cells()` – den returnerar tomma celler rad för rad)*


In [ ]:
# Övning 1: Spåra find_empty_cells
test_braede = [
    [1, 0, 0],
    [0, 2, 0],
    [0, 0, 1]
]

print("Spelplan:")
visa_braede(test_braede)

# Kör find_empty_cells och se vad den returnerar
tomma = find_empty_cells(test_braede)
print(f"Tomma celler: {tomma}")
print(f"DFS provar drag i denna ordning!")
print()
print("Gör DFS-drag:")
drag = dfs_agent(test_braede, 1)
print(f"DFS väljer: {drag}")


### Övning 2: Djupet i DFS-trädet

**Fråga:** Vad är det **maximala djupet** i DFS-trädet för Tre i rad?

*(Ledtråd: Hur många drag kan spelas totalt på en 3x3 spelplan?)*

Skriv ditt svar nedan:


In [ ]:
# Övning 2: Räkna djupet
spelplan_storlek = BOARD_SIZE * BOARD_SIZE
print(f"Spelplanens storlek: {BOARD_SIZE} x {BOARD_SIZE} = {spelplan_storlek} celler")
print(f"Maximalt antal drag: {spelplan_storlek}")
print(f"Därför är maximalt djup i DFS-trädet: {spelplan_storlek}")
print()
print("Men i praktiken slutar spelet ofta FÖRE det maximala djupet,")
print("när en spelare vinner!")


### 🟡 Medelsvåra övningar

### Övning 3: BFS och kön

**Fråga:** Vilken datastruktur använder BFS och varför?

Välj rätt alternativ och kör koden för att verifiera:
- A) Stack (LIFO – Last In, First Out)
- B) Kö (FIFO – First In, First Out)
- C) Lista (random access)


In [ ]:
# Övning 3: Testa kön vs stacken
from collections import deque

print("🧪 Demonstration av kö (BFS) vs stack (DFS):")
print()

# KÖ – FIFO (BFS använder detta)
kö = deque()
kö.append("Nivå 1 - Drag A")
kö.append("Nivå 1 - Drag B")
kö.append("Nivå 1 - Drag C")
print("KÖ (FIFO) – BFS:")
while kö:
    print(f"  Tar ut: {kö.popleft()}")
print()

# STACK – LIFO (DFS kan använda detta)
stack = []
stack.append("Nivå 1 - Drag A")
stack.append("Nivå 2 - Drag A1")
stack.append("Nivå 3 - Drag A1a")
print("STACK (LIFO) – DFS:")
while stack:
    print(f"  Tar ut: {stack.pop()}")

print()
print("💡 Kön (FIFO) ger oss BFS – vi utforskar brett!")
print("💡 Stacken (LIFO) ger oss DFS – vi utforskar djupt!")


### Övning 4: Begränsa DFS-djupet

Vad händer om vi begränsar DFS till djup 2?  
Modifiera `dfs_agent()` för att bara söka 2 nivåer djupt.


In [ ]:
# Övning 4: Begränsad DFS
def dfs_begransad(board, player, max_djup=2):
    """
    DFS med begränsat djup.
    Söker bara {max_djup} drag framåt.
    """
    def dfs(bräde, spelare, djup=0):
        vinnare = check_winner(bräde)
        if vinnare == player:
            return 1, None
        elif vinnare != 0 and vinnare != player:
            return -1, None
        elif vinnare == 3:
            return 0, None

        # 🔑 BEGRÄNSNING: Stoppa om vi nått max djup
        if djup >= max_djup:
            return 0, None  # Oavgjort om vi inte vet mer

        bästa_poäng = -2
        bästa_drag = None
        for (rad, kol) in find_empty_cells(bräde):
            bräde[rad][kol] = spelare
            nästa = 2 if spelare == 1 else 1
            poäng, _ = dfs(bräde, nästa, djup + 1)
            poäng = -poäng
            bräde[rad][kol] = 0
            if poäng > bästa_poäng:
                bästa_poäng = poäng
                bästa_drag = (rad, kol)
        return bästa_poäng, bästa_drag

    _, drag = dfs(board, player)
    return drag

# Jämför begränsad DFS mot vanlig DFS
print("⏱️  Mäter hastighet: Begränsad DFS (djup=2) vs Full DFS")
print()

start = time.time()
v1, v2, o = run_tournament(dfs_begransad, random_agent, 50)
tid_begransad = time.time() - start
print(f"Begränsad DFS (djup=2) vs Slump: {v1}V/{v2}F/{o}O | ⏱️ {tid_begransad:.2f}s")

start = time.time()
v1, v2, o = run_tournament(dfs_agent, random_agent, 50)
tid_full = time.time() - start
print(f"Full DFS            vs Slump: {v1}V/{v2}F/{o}O | ⏱️ {tid_full:.2f}s")

print()
print(f"💡 Full DFS är {tid_full/tid_begransad:.1f}x långsammare men {round(v1/50*100)}% effektivare!")


### 🔴 Avancerade övningar

### Övning 5: Jämförelse – Vad är skillnaden?

Kör koden nedan och analysera resultaten.  
**Frågor att svara på:**
1. Vilken algoritm vinner mest?
2. Vilken algoritm är snabbast?
3. Vilken algoritm skulle du välja om datorn var långsam?


In [ ]:
# Övning 5: Stor jämförelse
print("📊 STOR JÄMFÖRELSE – 100 spel vardera")
print("=" * 50)

resultat = {}

for namn, agent in [("DFS", dfs_agent), ("BFS", bfs_agent), ("Regelbaserad", rule_based_agent)]:
    start = time.time()
    v1, v2, o = run_tournament(agent, random_agent, 100)
    tid = time.time() - start
    resultat[namn] = {"vinster": v1, "tid": tid}
    print(f"{namn:15s}: {v1:3d} vinster | {tid:.2f}s totalt | {tid/100*1000:.1f}ms/spel")

print("=" * 50)
bäst_vinster = max(resultat, key=lambda x: resultat[x]["vinster"])
snabbast = min(resultat, key=lambda x: resultat[x]["tid"])
print(f"🏆 Flest vinster: {bäst_vinster}")
print(f"⚡ Snabbast: {snabbast}")
print()
print("Vilken väljer DU? Skriv din analys i nästa cell!")


In [ ]:
# ✏️ Skriv din analys här!
# Ersätt texten nedan med dina egna tankar

min_analys = """
Jag tycker att... [skriv här]

Den snabbaste algoritmen är... [skriv här]

Den bästa algoritmen är... [skriv här] eftersom...

Om datorn var långsam skulle jag välja... [skriv här] för att...
"""

print(min_analys)


## 📝 Quiz – Testa dina kunskaper!

Svara på frågorna nedan. Kör varje kodcell för att se om ditt svar är rätt!


In [ ]:
# ❓ Quiz 1: Vad returnerar find_empty_cells på ett tomt bräde?
# Välj ett alternativ: a, b eller c
# a) En siffra (antal tomma celler)
# b) En lista med (rad, kol)-par för alla 9 celler
# c) True om brädet är tomt

mitt_svar_q1 = "b"  # ← Ändra till ditt svar!

# Kör för att kontrollera
tomt_bräde = [[0]*3 for _ in range(3)]
resultat = find_empty_cells(tomt_bräde)
print(f"find_empty_cells returnerar: {resultat}")
print(f"Typ: {type(resultat)}")
print(f"Antal element: {len(resultat)}")
print()
if mitt_svar_q1 == "b":
    print("✅ Rätt! Den returnerar en lista med alla 9 (rad, kol)-par.")
else:
    print(f"❌ Ditt svar '{mitt_svar_q1}' är fel. Rätt svar är 'b'.")


In [ ]:
# ❓ Quiz 2: Vilken datastruktur är KÖ (som BFS använder)?
# a) LIFO (Last In, First Out)
# b) FIFO (First In, First Out)
# c) Slumpmässig ordning

mitt_svar_q2 = "b"  # ← Ändra till ditt svar!

print("En kö är som att köa i en affär:")
print("Den som kom FÖRST får hjälp FÖRST!")
print()
if mitt_svar_q2 == "b":
    print("✅ Rätt! En kö är FIFO – First In, First Out.")
    print("   I BFS lägger vi till nya tillstånd SIST och tar ut FÖRST.")
else:
    print(f"❌ Ditt svar '{mitt_svar_q2}' är fel. Rätt svar är 'b' (FIFO).")


In [ ]:
# ❓ Quiz 3: DFS returnerar -poäng (invertering). Varför?
# a) För att göra koden mer komplicerad
# b) För att bra drag för motståndaren är DÅLIGA för oss
# c) Det är ett slumpmässigt val

mitt_svar_q3 = "b"  # ← Ändra till ditt svar!

print("Tänk på det så här:")
print("Om motståndaren gör ett bra drag (poäng +1 för dem),")
print("är det dåligt för oss (poäng -1 för oss).")
print()
print("Därför: -poäng = vi inverterar perspektivet!")
print()
if mitt_svar_q3 == "b":
    print("✅ Rätt! Inverteringen byter perspektiv mellan spelarna.")
else:
    print(f"❌ Ditt svar '{mitt_svar_q3}' är fel. Rätt svar är 'b'.")


In [ ]:
# ❓ Quiz 4: Varför är BFS långsammare än DFS för Tre i rad?
# a) BFS har fler buggar i koden
# b) BFS måste hålla ALLA tillstånd på aktuell nivå i minnet (kön)
# c) BFS är alltid långsammare än DFS

mitt_svar_q4 = "b"  # ← Ändra till ditt svar!

print("BFS håller alla tillstånd på aktuell nivå i minnet.")
print()
print("Vid nivå 1: 9 tillstånd i kön")
print("Vid nivå 2: 9*8=72 tillstånd i kön")
print("Vid nivå 3: 72*7=504 tillstånd i kön")
print("... (växer snabbt!)")
print()
if mitt_svar_q4 == "b":
    print("✅ Rätt! BFS minne växer snabbt med djupet – det tar tid!")
else:
    print(f"❌ Ditt svar '{mitt_svar_q4}' är fel. Rätt svar är 'b'.")


In [ ]:
# ❓ Quiz 5: Vad heter nästa lektion och vad handlar den om?
# a) Lektion 6: Minimax – en algoritm som tänker som BÅDA spelarna
# b) Lektion 6: Neurala nätverk – AI som lär sig av erfarenhet
# c) Lektion 6: Random AI – förbättrad slumpmässig agent

mitt_svar_q5 = "a"  # ← Ändra till ditt svar!

print("DFS och BFS hittar vinnande drag, men de hanterar inte")
print("motståndarens MOTDRAG optimalt.")
print()
print("Minimax är lösningen – den tänker som BÅDA spelarna!")
print()
if mitt_svar_q5 == "a":
    print("✅ Rätt! I Lektion 6 lär vi oss Minimax – perfekt AI!")
else:
    print(f"❌ Ditt svar '{mitt_svar_q5}' är fel. Rätt svar är 'a'.")


## 🎯 Sammanfattning – Lektion 5

### Vad lärde vi oss idag?

| Begrepp | Förklaring |
|---------|------------|
| **Spelträd** | Träd av alla möjliga framtida spellägen |
| **DFS** | Djupet-först – utforskar ett sökväg helt innan nästa |
| **BFS** | Bredden-först – utforskar alla möjligheter på varje nivå |
| **Rekursion** | Funktion som anropar sig själv (används i DFS) |
| **Kö (deque)** | FIFO-datastruktur som BFS använder |
| **Backtrackning** | Att ångra ett drag och prova nästa möjlighet |

### ⚖️ DFS vs BFS – Jämförelse

| Egenskap | DFS | BFS |
|----------|-----|-----|
| Strategi | Djupt första | Brett första |
| Datastruktur | Stack (rekursion) | Kö (deque) |
| Minnesanvändning | Liten | Stor |
| Hittar kortaste vinst? | Inte alltid | ✅ Ja |
| Hastighet i Tre i rad | Snabb | Långsammare |

### 🤔 Problemet kvar

Både DFS och BFS hittar vinnande drag, men de hanterar inte  
motståndarens motdrag **optimalt**. De kan falla i fällor!

> 🔜 **Nästa lektion (Lektion 6): Minimax**  
> Minimax tänker som BÅDA spelarna och spelar **perfekt**!  
> Kan DU slå ett perfekt spelande AI? (Spoiler: Nej! 😄)

---

**Bra jobbat idag! 🌟 Ses i Lektion 6!**
